# 04 — Model Training

Train Random Forest on SMOTE-balanced features.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/final_features.csv')
FEAT_COLS = ['mean_dwell','median_flight','cv_flight','mean_del_freq','mean_tot_time']
df_m = df[FEAT_COLS + ['emotionIndex']].dropna()
print(f'Samples: {df_m.shape}')

## Encode labels + SMOTE balancing

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df_m['emotionIndex'])
X  = df_m[FEAT_COLS].values

print('Before SMOTE:', dict(zip(le.classes_, np.bincount(y))))
X_res, y_res = SMOTE(random_state=42, k_neighbors=3).fit_resample(X, y)
print('After SMOTE: ', dict(zip(le.classes_, np.bincount(y_res))))

## Cross-validation comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

models = {
    'Logistic Regression': Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(max_iter=1000,class_weight='balanced'))]),
    'Random Forest':       Pipeline([('sc',StandardScaler()),('clf',RandomForestClassifier(n_estimators=100,class_weight='balanced',random_state=42))]),
    'Gradient Boosting':   Pipeline([('sc',StandardScaler()),('clf',GradientBoostingClassifier(n_estimators=100,random_state=42))]),
    'SVM (RBF)':           Pipeline([('sc',StandardScaler()),('clf',SVC(kernel='rbf',class_weight='balanced',random_state=42))]),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    acc = cross_val_score(model, X_res, y_res, cv=skf, scoring='accuracy')
    f1  = cross_val_score(model, X_res, y_res, cv=skf, scoring='f1_macro')
    print(f'{name:22s}: acc={acc.mean():.4f} | f1={f1.mean():.4f}')

## Train final Random Forest (200 trees)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
best_model = Pipeline([('sc',StandardScaler()),('clf',RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42,n_jobs=-1))])
best_model.fit(X_tr, y_tr)
print(f'Test accuracy: {(best_model.predict(X_te)==y_te).mean():.4f}')

## Save artifacts

In [ ]:
artifacts = {'model': best_model, 'label_encoder': le, 'feature_names': FEAT_COLS, 'classes': le.classes_.tolist(), 'model_version': '1.0.0'}
with open('../model/mindtype_model.pkl','wb') as f:
    pickle.dump(artifacts, f)
with open('../model/encoder.pkl','wb') as f:
    pickle.dump(le, f)
print('Saved model artifacts')